# DLC Face Detection Training Pipeline

このNotebookは、GitHubで管理している顔検出モデルの学習コードを、Kaggle上で一括実行するための起動用Notebookです。

## このNotebookの役割

学習処理の本体は、`kaggle_line`ディレクトリ内のPythonファイルに分割して管理しています。

このNotebookは、`kaggle_line/cell_*.py`をファイル名順に読み込み、同じNotebook名前空間で実行します。

In [ ]:
# Clone and Run Training Pipeline
# ============================================================
#
# このセルの役割:
#
#   1. GitHubから指定したブランチ、タグ、コミットを取得する
#   2. kaggle_line内のPythonファイルを番号順に実行する
#   3. 学習から成果物出力までを完了する
#
# DatasetはKaggle Notebookへ追加済みのものを、
# Dataset作成セルが/kaggle/inputから自動検出します。
#
# ============================================================


from pathlib import Path
import os
import shutil
import subprocess


# ============================================================
# GitHub設定
# ============================================================

REPOSITORY_URL = (
    "https://github.com/Hiroyuki-Kobayashi-12/"
    "DLC-FaceDetection-PRE.git"
)

# ブランチ、タグ、または完全なコミットSHAを指定します。
GIT_REFERENCE = "sandbox/kobayashi"

PROJECT_DIRECTORY = Path(
    "/kaggle/working/DLC-FaceDetection-PRE"
)


# ============================================================
# GitHubからコードを取得
# ============================================================

if PROJECT_DIRECTORY.exists():
    shutil.rmtree(
        PROJECT_DIRECTORY
    )

subprocess.run(
    [
        "git",
        "clone",
        "--filter=blob:none",
        "--no-checkout",
        REPOSITORY_URL,
        str(PROJECT_DIRECTORY),
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(PROJECT_DIRECTORY),
        "fetch",
        "--depth",
        "1",
        "origin",
        GIT_REFERENCE,
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(PROJECT_DIRECTORY),
        "checkout",
        "--detach",
        "FETCH_HEAD",
    ],
    check=True,
)

os.chdir(
    PROJECT_DIRECTORY
)


# ============================================================
# 使用するGit情報を表示
# ============================================================

git_commit = subprocess.run(
    [
        "git",
        "-C",
        str(PROJECT_DIRECTORY),
        "rev-parse",
        "HEAD",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print(
    f"[GIT] Reference: {GIT_REFERENCE}"
)

print(
    f"[GIT] Commit: {git_commit}"
)

print(
    f"[GIT] Project: {PROJECT_DIRECTORY}"
)


# ============================================================
# 学習セルを取得
# ============================================================

CELL_DIRECTORY = (
    PROJECT_DIRECTORY
    / "kaggle_line"
)

CELL_FILES = sorted(
    CELL_DIRECTORY.glob(
        "cell_[0-9][0-9]_*.py"
    )
)

print(
    f"[PIPELINE] Cell count: {len(CELL_FILES)}"
)

for cell_file in CELL_FILES:
    print(
        f"[PIPELINE] Found: {cell_file.name}"
    )


# ============================================================
# 学習パイプラインを実行
# ============================================================

for cell_file in CELL_FILES:

    print()
    print(
        f"========== RUN {cell_file.name} =========="
    )

    cell_code = cell_file.read_text(
        encoding="utf-8"
    )

    exec(
        compile(
            cell_code,
            str(cell_file),
            "exec",
        ),
        globals(),
    )


# ============================================================
# 出力結果を確認
# ============================================================

OUTPUT_DIRECTORY = Path(
    "/kaggle/working/dlc26_outputs"
)

print()
print(
    "========== OUTPUT FILES =========="
)

for output_path in sorted(
    OUTPUT_DIRECTORY.rglob("*")
):
    if output_path.is_file():

        size_mb = (
            output_path.stat().st_size
            / 1024
            / 1024
        )

        print(
            f"{output_path.relative_to(OUTPUT_DIRECTORY)} "
            f"({size_mb:.2f} MB)"
        )